# Exercise 1 – Example: image → JSON → table

This notebook shows the whole way from one picture to a scored table, with as little code as possible. **Run every cell and read the code**. Then open `image_extraction_tasks.ipynb`: there you build the same steps yourself for new images and new fields – copy the code from here and adapt it.

1. **One image:** send a picture to the OpenAI API, read the answer and its cost.
2. **Fixed format:** get the answer as fields instead of free text.
3. **Several images:** collect the answers in a table.
4. **Score:** compare the table with hand labels.

## Setup

Select the **Python 3.11** kernel. The OpenAI API key comes from your `OPENAI_API_KEY` Codespaces secret. This notebook uses the **OpenAI API** (model `gpt-6-luna`); other providers, e.g. Google Gemini or Anthropic Claude, have their own APIs with different code.

In [ ]:
# Python libraries.
import sys
from typing import Literal

import pandas as pd
from openai import OpenAI
from pydantic import BaseModel
from sklearn.metrics import accuracy_score

# helper.py is in the main folder, one level up.
sys.path.append("..")
from helper import DATA, MODEL, ask, image_to_input, show_images

## 1. One image

A request is a **message** with a list of parts: here a text part and an image part. `image_to_input()` from `helper.py` turns the image file into such a part.

In [ ]:
image_path = DATA / "images" / "lake.jpg"
show_images([image_path])

In [ ]:
client = OpenAI()  # reads OPENAI_API_KEY

content = [
    {"type": "input_text", "text": "Describe this image in two sentences."},
    image_to_input(image_path),
]
response = client.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": content}],
    reasoning={"effort": "none"},  # no extra thinking: faster, cheaper
)
print(response.output_text)

**Cost:** every answer reports its tokens in `response.usage`. You pay per 1 million tokens, separately for input (question and image) and output (answer). The price depends on the model – look it up on its model page, [GPT-6 Luna model and pricing](https://developers.openai.com/api/docs/models/gpt-6-luna), or in the [OpenAI API pricing](https://developers.openai.com/api/docs/pricing) list.

In [ ]:
price_in, price_out = 0.10, 0.50  # US$ per 1 million tokens, gpt-6-luna

tokens_in = response.usage.input_tokens
tokens_out = response.usage.output_tokens
cost = (tokens_in * price_in + tokens_out * price_out) / 1_000_000
print(f"{tokens_in} + {tokens_out} tokens = US$ {cost:.6f} – for 1,000 images: US$ {cost * 1000:.2f}")

## 2. Fixed format

Free text is hard to compare. A Python class defines the fields of the answer; `ask()` from `helper.py` sends the prompt and the image like above and returns a dict with exactly these fields.

In [ ]:
class LakeRecord(BaseModel):
    scene: Literal["lake", "parking lot", "alpine pasture", "city street"]
    objects: list[str]


PROMPT = """
Pick the scene from the list.
Name up to five prominent objects as lowercase singular nouns.
"""

ask(PROMPT, LakeRecord, [image_path])

## 3. Several images → table

One row per image. We add the filename ourselves – the model could misspell it.

In [ ]:
files = ["lake.jpg", "lake_night.jpg", "lake_swan.jpg"]

rows = []
for name in files:
    row = ask(PROMPT, LakeRecord, [DATA / "images" / name])
    row["file"] = name
    rows.append(row)

table = pd.DataFrame(rows)
table

## 4. Score against hand labels

`labels.csv` has one row per image. The column `file` joins model answers (`_pred`) and labels (`_ref`).

In [ ]:
labels = pd.read_csv(DATA / "images" / "labels.csv")
scores = table.merge(labels, on="file", suffixes=("_pred", "_ref"))

print("Scene accuracy:", accuracy_score(scores["scene_ref"], scores["scene_pred"]))
scores[["file", "scene_pred", "scene_ref", "objects_pred", "objects_ref"]]

**Next:** open `image_extraction_tasks.ipynb` and build these steps yourself.

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')